In [28]:
import pandas as pd
import re

# Définir les chemins vers les fichiers Excel
jikan_data_path = 'F:/1.Boulot/01_Etudes/05_Analyses_Mangas/01_Datas/01_Jikan/mangas_jikan_data.xlsx'
anilist_data_path = 'F:/1.Boulot/01_Etudes/05_Analyses_Mangas/01_Datas/02_AniList/mangas_anilist_data.xlsx'

# Charger les fichiers avec les colonnes nécessaires
jikan_df = pd.read_excel(jikan_data_path, usecols=['title_english'])
anilist_df = pd.read_excel(anilist_data_path, usecols=['English Title'])

# Fonction de nettoyage des titres
def clean_titles(df, column_name, new_column_name):
    df[new_column_name] = (
        df[column_name]
        .str.upper()
        .str.replace(r'[^A-Z0-9 ]', '', regex=True)  # Conserve lettres, chiffres et espaces
        .str.strip()
    )
    df['initial'] = df[new_column_name].str[0]  # Extraire la première lettre
    return df[[column_name, new_column_name, 'initial']]

# Nettoyer les données
jikan_df = clean_titles(jikan_df, 'title_english', 'jikan_title_cleaned')
anilist_df = clean_titles(anilist_df, 'English Title', 'anilist_title_cleaned')

# Traiter les données pour chaque lettre et générer les DataFrames
for letter in map(chr, range(65, 91)):  # Lettres de 'A' à 'Z'
    # Filtrer les données par initiale
    jikan_subset = jikan_df[jikan_df['initial'] == letter]
    anilist_subset = anilist_df[anilist_df['initial'] == letter]
    
    # Fusionner les données sur les titres nettoyés
    merged_df = pd.merge(
        jikan_subset, anilist_subset,
        left_on='jikan_title_cleaned', right_on='anilist_title_cleaned',
        how='outer'
    )
    
    # Créer les colonnes finales
    final_df = pd.DataFrame({
        'All Titles': merged_df['jikan_title_cleaned'].combine_first(merged_df['anilist_title_cleaned']),
        'Jikan Titles': merged_df['title_english'],
        'AniList Titles': merged_df['English Title']
    })
    
    # Trier par ordre alphabétique sur la colonne 'All Titles'
    final_df = final_df.sort_values(by='All Titles').reset_index(drop=True)
    
    # Assigner le DataFrame à une variable dynamique
    globals()[f'mangas_{letter}'] = final_df

# Exemple d'affichage pour la lettre A
print("DataFrame pour la lettre A :")
print(mangas_A.head())


DataFrame pour la lettre A :
                                          All Titles  \
0                   A 19YEAROLD HUSBAND HAS A SECRET   
1  A 1LDK THAT COMES WITH A SUPERSADISTIC PRINCE ...   
2                                                A A   
3                                          A A PRIME   
4                                  A BABY OF HIS OWN   

                         Jikan Titles  \
0  A 19-Year-Old Husband Has a Secret   
1                                 NaN   
2                                 NaN   
3                          A, A Prime   
4                   A Baby of His Own   

                                      AniList Titles  
0                                                NaN  
1  A 1LDK that comes with a Super-Sadistic Prince...  
2                                              A, A'  
3                                                NaN  
4                                                NaN  


In [19]:
import requests
from bs4 import BeautifulSoup

# URL de la page de la série
url = 'https://www.manga-news.com/index.php/serie/Rodeurs-de-la-nuit-les'

# Envoyer une requête GET pour récupérer le contenu HTML
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Récupérer le titre de la série
    title = soup.select_one('h1.entry-page-title').text.strip() if soup.select_one('h1.entry-page-title') else "Titre non trouvé"
    
    # Récupérer le Titre VO sans le préfixe
    original_title_element = soup.select_one('li.title-vo')
    original_title = original_title_element.text.replace('Titre VO:', '').strip() if original_title_element else "Titre VO non trouvé"
    
    # Récupérer le Titre traduit sans le préfixe
    translated_title_element = soup.select_one('li.trad')
    translated_title = translated_title_element.text.replace('Titre traduit:', '').strip() if translated_title_element else "Titre traduit non trouvé"
    
    # Récupérer le résumé
    summary_element = soup.select_one('div.bigsize')
    if summary_element:
        summary = ''.join([line.strip() for line in summary_element.stripped_strings])
    else:
        summary = "Résumé non trouvé"
    
    # Afficher les informations récupérées
    print(f"Titre : {title}")
    print(f"Titre VO : {original_title}")
    print(f"Titre traduit : {translated_title}")
    print(f"Résumé : {summary}")

else:
    print(f"Erreur lors de l'accès à la page : {response.status_code}")


Titre : Demon Slayer
Titre VO : 鬼滅の刃
Titre traduit : Kimetsu no Yaiba
Résumé : Le Japon, au début du XXe siecle.Un petit marchand de charbon nommé Tanjiro vit une vie sans histoire dans les montagnes. Jusqu’au jour tragique où, après une courte absence, il retrouve son village et sa famille massacrés par un ogre ! La seule survivante de cette tragédie est sa jeune sœur Nezuko. Hélas, au contact de la bête, celle-ci s’est à son tour métamorphosée en monstre...Afin de renverser le processus et de venger sa famille, Tanjiro décide de partir en quête de vérité. Pour le jeune héros et sa sœur, c’est une longue aventure de sang et d’acier qui commence !


In [22]:

import requests
from bs4 import BeautifulSoup

# URL de la page de la série
url = 'https://www.manga-news.com/index.php/serie/Death-note'

# Envoyer une requête GET pour récupérer le contenu HTML
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Récupérer le titre de la série
    title = soup.select_one('h1.entry-page-title').text.strip() if soup.select_one('h1.entry-page-title') else "Titre non trouvé"
    
    # Récupérer le Titre VO sans le préfixe
    original_title_element = soup.select_one('li.title-vo')
    original_title = original_title_element.text.replace('Titre VO:', '').strip() if original_title_element else "Titre VO non trouvé"
    
    # Récupérer le Titre traduit sans le préfixe
    translated_title_element = soup.select_one('li.trad')
    translated_title = translated_title_element.text.replace('Titre traduit:', '').strip() if translated_title_element else "Titre traduit non trouvé"
    
    # Récupérer le résumé
    summary_element = soup.select_one('div.bigsize')
    if summary_element:
        summary = ''.join([line.strip() for line in summary_element.stripped_strings])
    else:
        summary = "Résumé non trouvé"
    
    # Afficher les informations récupérées
    print(f"Titre : {title}")
    print(f"Titre VO : {original_title}")
    print(f"Titre traduit : {translated_title}")
    print(f"Résumé : {summary}")

else:
    print(f"Erreur lors de l'accès à la page : {response.status_code}")


Titre : Death Note
Titre VO : デスノート
Titre traduit : Death note
Résumé : Light Yagami est un lycéen âgé de 17 ans, jeune homme brillant, fils d'un policier, il découvre un étrange carnet qui se révèle être le livre d'un dieu de la mort : Ryûk ! Light apprendra vite quels terribles pouvoirs renferment ce carnet : tous ceux dont le nom est inscrit dans le Death Note sont appelés à  mourir dans les 40 secondes qui suivent !Les implications sont énormes et en possession d'un tel carnet Light est potentiellement capable d'imposer sa loi à  un monde qu'il estime perverti. Mais peut-on choisir qui va vivre et qui va mourir ? Certaines personnes méritent-elles de mourir par la seule volonté d'un adolescent, à  la fois juge et bourreau pour une sentence irrévocable ?En agissant de la sorte Light devient lui-même un criminel, il éveille ainsi l'attention de L, enquêteur mystérieux mandaté par Interpol. Un duel sans merci s'engage entre ces deux esprits exceptionnels !


In [31]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL de la page listant les mangas par ordre alphabétique pour la lettre D
listing_url = 'https://www.manga-news.com/index.php/series/D'

# Liste pour stocker les données
data = []

# Envoyer une requête GET pour récupérer le contenu HTML
response = requests.get(listing_url)
if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Trouver tous les éléments de liste de mangas
    manga_rows = soup.select('tr')  # Sélectionner chaque ligne de tableau
    
    # Parcourir chaque manga et extraire les informations
    for row in manga_rows:
        title_element = row.select_one('span.item-list-content-title')
        author_element = row.select_one('td.list-1-data a[title="Dessinateur"]')
        editor_element = row.select('td.list-1-data')[1] if len(row.select('td.list-1-data')) > 1 else None
        volumes_element = row.select('td.list-1-data')[2] if len(row.select('td.list-1-data')) > 2 else None
        genre_element = row.select('td.list-1-data')[3] if len(row.select('td.list-1-data')) > 3 else None
        link_element = row.select_one('a[title]')  # Sélectionner le premier lien avec un attribut `title`

        # Extraire le texte de chaque élément ou définir un message si absent
        title = title_element.text.strip() if title_element else "Titre non trouvé"
        author = author_element.text.strip() if author_element else "Auteur non trouvé"
        editor = editor_element.text.strip() if editor_element else "Éditeur non trouvé"
        volumes = volumes_element.text.strip() if volumes_element else "Nombre de volumes non trouvé"
        genre = genre_element.text.strip() if genre_element else "Genre non trouvé"
        link = link_element['href'] if link_element else "Lien non trouvé"
        
        # Ajouter les données dans la liste
        data.append({
            'Titre': title,
            'Auteur': author,
            'Éditeur': editor,
            'Nombre de volumes': volumes,
            'Genre': genre,
            'URL': link
        })

# Convertir les données en DataFrame et afficher les résultats
df = pd.DataFrame(data)
print(df)


                                Titre             Auteur              Éditeur  \
0                    Titre non trouvé  Auteur non trouvé   Éditeur non trouvé   
1    D'Arc - Histoire de Jeanne D'arc      KONDO Katsuya   Black Box Editions   
2                   D'encre et de feu      SHIZUHA / KTA                  H2T   
3                    D'un amour brisé               LALA              Bontoon   
4                                 D'v      Takuya FUJIMA               Panini   
..                                ...                ...                  ...   
665        Dévoreur de souvenirs (le)   Nachiyo MURAYAMA    Delcourt / Tonkam   
666         Dîner de la sorcière (le)   Rumiko TAKAHASHI    Delcourt / Tonkam   
667                Dôgen - maître zen    HISAMATSU Fumio                Sully   
668       Dôjinshi - Guilt ! Pleasure              TogaQ         Taifu comics   
669            Dômu - Rêves d'enfants    Katsuhiro OTOMO  Humanoides associes   

                Nombre de v

In [33]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Exemple de DataFrame avec une colonne "URL" contenant les liens vers les mangas
# df = pd.read_csv('mangas_D_listing.csv')  # Chargez votre DataFrame existant

# Initialiser les nouvelles colonnes
df['Titre Série'] = ""
df['Titre VO'] = ""
df['Titre traduit'] = ""
df['Résumé'] = ""

# Fonction pour récupérer les informations à partir de l'URL
def fetch_manga_details(url):
    if url == "Lien non trouvé" or not url.startswith("https"):
        return "URL invalide", "URL invalide", "URL invalide", "URL invalide"
    
    response = requests.get(url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Récupérer le titre de la série
        title = soup.select_one('h1.entry-page-title').text.strip() if soup.select_one('h1.entry-page-title') else "Titre non trouvé"
        
        # Récupérer le Titre VO sans le préfixe
        original_title_element = soup.select_one('li.title-vo')
        original_title = original_title_element.text.replace('Titre VO:', '').strip() if original_title_element else "Titre VO non trouvé"
        
        # Récupérer le Titre traduit sans le préfixe
        translated_title_element = soup.select_one('li.trad')
        translated_title = translated_title_element.text.replace('Titre traduit:', '').strip() if translated_title_element else "Titre traduit non trouvé"
        
        # Récupérer le résumé
        summary_element = soup.select_one('div.bigsize')
        if summary_element:
            summary = ''.join([line.strip() for line in summary_element.stripped_strings])
        else:
            summary = "Résumé non trouvé"
        
        return title, original_title, translated_title, summary
    else:
        return "Erreur", "Erreur", "Erreur", "Erreur"

# Boucler sur chaque URL dans le DataFrame et remplir les nouvelles colonnes
for index, row in df.iterrows():
    title, original_title, translated_title, summary = fetch_manga_details(row['URL'])
    df.at[index, 'Titre Série'] = title
    df.at[index, 'Titre VO'] = original_title
    df.at[index, 'Titre traduit'] = translated_title
    df.at[index, 'Résumé'] = summary

# Afficher le DataFrame avec les nouvelles informations
print(df)



                                Titre             Auteur              Éditeur  \
0                    Titre non trouvé  Auteur non trouvé   Éditeur non trouvé   
1    D'Arc - Histoire de Jeanne D'arc      KONDO Katsuya   Black Box Editions   
2                   D'encre et de feu      SHIZUHA / KTA                  H2T   
3                    D'un amour brisé               LALA              Bontoon   
4                                 D'v      Takuya FUJIMA               Panini   
..                                ...                ...                  ...   
665        Dévoreur de souvenirs (le)   Nachiyo MURAYAMA    Delcourt / Tonkam   
666         Dîner de la sorcière (le)   Rumiko TAKAHASHI    Delcourt / Tonkam   
667                Dôgen - maître zen    HISAMATSU Fumio                Sully   
668       Dôjinshi - Guilt ! Pleasure              TogaQ         Taifu comics   
669            Dômu - Rêves d'enfants    Katsuhiro OTOMO  Humanoides associes   

                Nombre de v

In [35]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# URL de la page listant les mangas par ordre alphabétique pour la lettre D
listing_url = 'https://www.manga-news.com/index.php/series/D'

# Liste pour stocker les données de la page de listing
data = []

# Envoyer une requête GET pour récupérer le contenu HTML
response = requests.get(listing_url)
if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Trouver tous les éléments de liste de mangas
    manga_rows = soup.select('tr')
    
    # Parcourir chaque manga et extraire les informations de base
    for row in manga_rows:
        title_element = row.select_one('span.item-list-content-title')
        author_element = row.select_one('td.list-1-data a[title="Dessinateur"]')
        editor_element = row.select('td.list-1-data')[1] if len(row.select('td.list-1-data')) > 1 else None
        volumes_element = row.select('td.list-1-data')[2] if len(row.select('td.list-1-data')) > 2 else None
        genre_element = row.select('td.list-1-data')[3] if len(row.select('td.list-1-data')) > 3 else None
        link_element = row.select_one('a[title]')

        # Extraire le texte de chaque élément ou définir un message si absent
        title = title_element.text.strip() if title_element else "Titre non trouvé"
        author = author_element.text.strip() if author_element else "Auteur non trouvé"
        editor = editor_element.text.strip() if editor_element else "Éditeur non trouvé"
        volumes = volumes_element.text.strip() if volumes_element else "Nombre de volumes non trouvé"
        genre = genre_element.text.strip() if genre_element else "Genre non trouvé"
        link = link_element['href'] if link_element else "Lien non trouvé"
        
        # Ajouter les données dans la liste
        data.append({
            'Titre': title,
            'Auteur': author,
            'Éditeur': editor,
            'Nombre de volumes': volumes,
            'Genre': genre,
            'URL': link
        })

# Convertir les informations de base en DataFrame
df = pd.DataFrame(data)

# Initialiser les nouvelles colonnes pour les détails supplémentaires
df['Titre Série'] = ""
df['Titre VO'] = ""
df['Titre traduit'] = ""
df['Résumé'] = ""
df['Origine'] = ""

# Fonction pour récupérer les détails à partir de l'URL de chaque manga
def fetch_manga_details(url):
    if url == "Lien non trouvé" or not url.startswith("https"):
        return "URL invalide", "URL invalide", "URL invalide", "URL invalide", "URL invalide"
    
    response = requests.get(url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Récupérer le titre de la série
        title = soup.select_one('h1.entry-page-title').text.strip() if soup.select_one('h1.entry-page-title') else "Titre non trouvé"
        
        # Récupérer le Titre VO sans le préfixe
        original_title_element = soup.select_one('li.title-vo')
        original_title = original_title_element.text.replace('Titre VO:', '').strip() if original_title_element else "Titre VO non trouvé"
        
        # Récupérer le Titre traduit sans le préfixe
        translated_title_element = soup.select_one('li.trad')
        translated_title = translated_title_element.text.replace('Titre traduit:', '').strip() if translated_title_element else "Titre traduit non trouvé"
        
        # Récupérer le résumé
        summary_element = soup.select_one('div.bigsize')
        summary = ''.join([line.strip() for line in summary_element.stripped_strings]) if summary_element else "Résumé non trouvé"
        
        # Récupérer l'origine
        origin_element = soup.find('strong', string="Origine")
        origin = origin_element.find_next_sibling(string=True).strip() if origin_element else "Origine non trouvée"
        
        return title, original_title, translated_title, summary, origin
    else:
        return "Erreur", "Erreur", "Erreur", "Erreur", "Erreur"

# Étape 2 : Traiter le DataFrame par blocs pour récupérer les détails supplémentaires
chunk_size = 50  # Nombre de lignes à traiter par bloc
for i in range(0, len(df), chunk_size):
    chunk = df.iloc[i:i + chunk_size]
    for index, row in chunk.iterrows():
        title, original_title, translated_title, summary, origin = fetch_manga_details(row['URL'])
        df.at[index, 'Titre Série'] = title
        df.at[index, 'Titre VO'] = original_title
        df.at[index, 'Titre traduit'] = translated_title
        df.at[index, 'Résumé'] = summary
        df.at[index, 'Origine'] = origin
    time.sleep(1)  # Pause pour éviter de surcharger le serveur

# Afficher le DataFrame avec les nouvelles informations
print(df)



                                Titre             Auteur              Éditeur  \
0                    Titre non trouvé  Auteur non trouvé   Éditeur non trouvé   
1    D'Arc - Histoire de Jeanne D'arc      KONDO Katsuya   Black Box Editions   
2                   D'encre et de feu      SHIZUHA / KTA                  H2T   
3                    D'un amour brisé               LALA              Bontoon   
4                                 D'v      Takuya FUJIMA               Panini   
..                                ...                ...                  ...   
665        Dévoreur de souvenirs (le)   Nachiyo MURAYAMA    Delcourt / Tonkam   
666         Dîner de la sorcière (le)   Rumiko TAKAHASHI    Delcourt / Tonkam   
667                Dôgen - maître zen    HISAMATSU Fumio                Sully   
668       Dôjinshi - Guilt ! Pleasure              TogaQ         Taifu comics   
669            Dômu - Rêves d'enfants    Katsuhiro OTOMO  Humanoides associes   

                Nombre de v